In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType
from pyspark.sql.functions import col, when, current_date, sum as _sum, desc

spark = SparkSession.builder \
    .appName("Spark Assignment Lesson 13") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "matrix") \
    .config("spark.hadoop.fs.s3a.secret.key", "matrix123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

In [2]:
cust_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("signup_date", StringType(), True),
    StructField("balance", DoubleType(), True),
    StructField("vip_status", StringType(), True)
])

card_schema = StructType([
    StructField("transaction_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("txn_date", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("merchant", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("status", StringType(), True)
])

In [3]:
df_cust = spark.read.csv("cust.csv", header=True, schema=cust_schema)
df_card = spark.read.csv("card_trn.csv", header=True, schema=card_schema)

In [4]:
df_cust.show(10)

+-----------+------+----+-----------+-------+----------+
|customer_id|  name| age|signup_date|balance|vip_status|
+-----------+------+----+-----------+-------+----------+
|          1| Nazim|  25| 2024-01-10| 1000.5|      true|
|          2| Samir|NULL| 2024-02-15|  200.0|     false|
|          3|Shahin|  30| 2024-03-20| 350.75|      true|
|          4| Aysel|NULL| 2024-04-05| 1500.0|      true|
|          5|  Rauf|  28| 2024-05-01|   NULL|     false|
|          6| Murad|  32| 2024-06-10|  950.0|      true|
|          7| Leyla|  22| 2024-07-25| 1200.0|     false|
|          8| Orxan|  45| 2024-08-19|  800.1|      true|
|          9| Fidan|  29| 2024-09-01|   NULL|     false|
|         10| Tural|  40| 2024-10-11|  700.0|      true|
+-----------+------+----+-----------+-------+----------+



In [5]:
df_card.show(10)

+--------------+-----------+----------+------+----------+-------+--------+-------+
|transaction_id|customer_id|  txn_date|amount|  merchant|channel|currency| status|
+--------------+-----------+----------+------+----------+-------+--------+-------+
|          2001|          1|2024-10-01|150.25|    Amazon| ONLINE|     USD|SUCCESS|
|          2002|          2|2024-10-02|  75.0|      Zara|    POS|     AZN|SUCCESS|
|          2003|          3|2024-10-03| 300.1|     Apple| ONLINE|     USD| FAILED|
|          2004|          4|2024-10-04| 50.75|      Bolt| ONLINE|     AZN|SUCCESS|
|          2005|          5|2024-10-05| 120.0|      Uber| ONLINE|     AZN|SUCCESS|
|          2006|          6|2024-10-06| 220.4|AliExpress| ONLINE|     USD|SUCCESS|
|          2007|          7|2024-10-07|  80.0|     Bravo|    POS|     AZN|SUCCESS|
|          2008|          8|2024-10-08| 400.0|   Samsung| ONLINE|     USD|SUCCESS|
|          2009|          9|2024-10-09|  60.0|      Araz|    POS|     AZN| FAILED|
|   

In [6]:
df_cust_clean = df_cust.fillna(0, subset=["balance"])

In [7]:
df_cust_final = df_cust_clean \
    .withColumn("age_group",
                when(col("age") < 25, "young")
                .when((col("age") >= 25) & (col("age") <= 40), "adult")
                .otherwise("senior")) \
    .withColumnRenamed("vip_status", "flg_is_vip") \
    .withColumn("insert_date", current_date())

In [8]:
df_cust_final.show(10)

+-----------+------+----+-----------+-------+----------+---------+-----------+
|customer_id|  name| age|signup_date|balance|flg_is_vip|age_group|insert_date|
+-----------+------+----+-----------+-------+----------+---------+-----------+
|          1| Nazim|  25| 2024-01-10| 1000.5|      true|    adult| 2026-02-17|
|          2| Samir|NULL| 2024-02-15|  200.0|     false|   senior| 2026-02-17|
|          3|Shahin|  30| 2024-03-20| 350.75|      true|    adult| 2026-02-17|
|          4| Aysel|NULL| 2024-04-05| 1500.0|      true|   senior| 2026-02-17|
|          5|  Rauf|  28| 2024-05-01|    0.0|     false|    adult| 2026-02-17|
|          6| Murad|  32| 2024-06-10|  950.0|      true|    adult| 2026-02-17|
|          7| Leyla|  22| 2024-07-25| 1200.0|     false|    young| 2026-02-17|
|          8| Orxan|  45| 2024-08-19|  800.1|      true|   senior| 2026-02-17|
|          9| Fidan|  29| 2024-09-01|    0.0|     false|    adult| 2026-02-17|
|         10| Tural|  40| 2024-10-11|  700.0|      t

In [9]:
df_cust_final.write \
        .format("parquet") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save("s3a://silver/customers_processed")

In [11]:
df_cust_final.createOrReplaceTempView("customers")
df_card.createOrReplaceTempView("transactions")

sql_query = """
    SELECT 
        c.customer_id,
        c.name,
        SUM(t.amount) as total_amount
    FROM transactions t
    JOIN customers c ON t.customer_id = c.customer_id
    GROUP BY c.customer_id, c.name
    ORDER BY total_amount DESC
    LIMIT 2
"""

top_2_sql = spark.sql(sql_query)

In [12]:
top_2_sql.show()

+-----------+------+------------+
|customer_id|  name|total_amount|
+-----------+------+------------+
|          3|Shahin|      800.85|
|          8| Orxan|       580.0|
+-----------+------+------------+

